# 🧠 Model — Treino, F1 *pooled* vs *LODO* e Grad-CAM

Projeto 7 (**Imagem**) · PAD 2026 (UFG) · fase **M (Model)**

O que este notebook entrega (os resultados da apresentação):
1. Treina um classificador (**transfer learning**, ResNet18) nas 5 classes;
2. **F1-macro *pooled*** (fontes misturadas) — visão otimista;
3. **F1-macro *LODO*** (treina em N−1 fontes, testa na restante) — visão realista;
4. A diferença = o **gap de generalização** (a hipótese do projeto);
5. **Grad-CAM** mostrando se o modelo olhou a lesão ou o fundo.

> Pré-requisito: ter rodado o **Get** e salvo o dataset no Drive. Use **GPU** (Ambiente → GPU).

## 1. Montar o Drive e carregar o dataset + manifesto

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import pandas as pd, os, glob
os.chdir('/content')
os.makedirs('data', exist_ok=True)          # garante que data/ existe
!rm -rf data/derived
!cp -r /content/drive/MyDrive/banana_pad/derived data/derived
!cp /content/drive/MyDrive/banana_pad/manifest.csv manifest.csv
assert os.path.isdir('data/derived'), 'derived nao copiou do Drive — rode o Get (01_get) e confira a celula 14'
df = pd.read_csv('manifest.csv')
df = df[(df['dup_of'].fillna('') == '') & (df['note'].fillna('') != 'label_conflict')].reset_index(drop=True)
CLASSES = sorted(df['label'].unique()); idx = {c: i for i, c in enumerate(CLASSES)}
print(len(df), 'imagens |', len(glob.glob('data/derived/**/*.jpg', recursive=True)), 'arquivos |', CLASSES)

## 2. Dataset e transformações
Treino com *augmentation* leve; validação/teste determinístico. Normalização ImageNet (a mesma para todas as fontes — regra do LODO).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.2, 0.2, 0.2, 0.05), transforms.ToTensor(), NORM])
tf_eval = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), NORM])
class BananaDS(Dataset):
    def __init__(self, frame, tf): self.f = frame.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        return self.tf(Image.open(r['image_id']).convert('RGB')), idx[r['label']]

## 3. Funções de modelo, treino e avaliação
Um helper reutilizado tanto no *pooled* quanto no *LODO* (transfer learning: backbone ImageNet + cabeça nova).

In [ ]:
from torchvision import models
from sklearn.metrics import f1_score
import numpy as np
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
def make_model(n):
    m = models.resnet18(weights='IMAGENET1K_V1')
    m.fc = torch.nn.Linear(m.fc.in_features, n)
    return m.to(DEV)
def train(m, dl, epochs=5, lr=1e-3):
    opt = torch.optim.Adam(m.parameters(), lr=lr); crit = torch.nn.CrossEntropyLoss()
    m.train()
    for e in range(epochs):
        for x, y in dl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(); loss = crit(m(x), y); loss.backward(); opt.step()
        print(f'  epoca {e+1}/{epochs} ok')
    return m
def predict(m, dl):
    m.eval(); ys, ps, pr = [], [], []
    with torch.no_grad():
        for x, y in dl:
            out = m(x.to(DEV)).softmax(1).cpu().numpy()    # probabilidades
            pr.append(out); ps += out.argmax(1).tolist(); ys += y.tolist()
    return np.array(ys), np.array(ps), np.concatenate(pr)   # rótulos, preds, probs

## 4. Treino *pooled* (todas as fontes misturadas)
Split estratificado por classe (80/20). Este é o número **otimista**.

In [ ]:
from sklearn.model_selection import train_test_split
tr, va = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=0)
dl_tr = DataLoader(BananaDS(tr, tf_train), batch_size=32, shuffle=True, num_workers=2)
dl_va = DataLoader(BananaDS(va, tf_eval), batch_size=32, num_workers=2)
model = make_model(len(CLASSES))
train(model, dl_tr, epochs=5)

## 5. Avaliar *pooled* — F1-macro + matriz de confusão

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y, p, prob = predict(model, dl_va)
f1_pooled = f1_score(y, p, average='macro')
print('F1-macro POOLED =', round(f1_pooled, 3))
print(classification_report(y, p, target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y, p, display_labels=CLASSES, xticks_rotation=45); plt.show()

### Sensibilidade, especificidade e ROC/AUC (pooled)
As métricas que o prof destacou no vídeo: **sensibilidade** (recall — acerta os doentes),
**especificidade** (acerta os saudáveis) e a **AUC** (área sob a curva ROC) por classe —
robustas para classes desbalanceadas. A curva ROC mostra o trade-off entre as duas.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
yb = np.eye(len(CLASSES))[y]                          # rótulos verdadeiros em one-hot
C = confusion_matrix(y, p, labels=range(len(CLASSES)))
print(f"{'classe':<16}{'sensib.':>9}{'especif.':>10}{'AUC':>7}")
for i, c in enumerate(CLASSES):
    TP = C[i, i]; FN = C[i].sum() - TP; FP = C[:, i].sum() - TP; TN = C.sum() - TP - FN - FP
    sens = TP / (TP + FN) if TP + FN else float('nan')   # sensibilidade = recall
    spec = TN / (TN + FP) if TN + FP else float('nan')   # especificidade
    auc = roc_auc_score(yb[:, i], prob[:, i]) if yb[:, i].sum() else float('nan')
    print(f"{c:<16}{sens:>9.2f}{spec:>10.2f}{auc:>7.2f}")
print('AUC macro (OvR) =', round(roc_auc_score(y, prob, multi_class='ovr', average='macro'), 3))
plt.figure(figsize=(6, 5))
for i, c in enumerate(CLASSES):
    if yb[:, i].sum() == 0: continue
    fpr, tpr, _ = roc_curve(yb[:, i], prob[:, i]); plt.plot(fpr, tpr, label=c)
plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
plt.xlabel('1 - especificidade (FPR)'); plt.ylabel('sensibilidade (TPR)')
plt.title('Curvas ROC por classe (OvR)'); plt.legend(fontsize=7); plt.show()

## 6. Avaliação *LODO* (leave-one-dataset-out)
Para cada fonte: treina nas **outras** e testa nela. O F1-macro é calculado **só nas classes
presentes no treino** daquele fold (classes de fonte única não penalizam injustamente).

In [ ]:
fontes = sorted(df['source'].unique()); lodo = {}
for s in fontes:
    tr_s, te_s = df[df.source != s], df[df.source == s]
    presentes = sorted(tr_s['label'].unique())                 # classes vistas no treino
    te_s = te_s[te_s['label'].isin(presentes)]
    if len(te_s) == 0: continue
    print(f'LODO — deixando {s} de fora (treino={len(tr_s)}, teste={len(te_s)})')
    m = make_model(len(CLASSES))
    train(m, DataLoader(BananaDS(tr_s, tf_train), batch_size=32, shuffle=True, num_workers=2), epochs=5)
    y, p, _ = predict(m, DataLoader(BananaDS(te_s, tf_eval), batch_size=32, num_workers=2))
    labs = [idx[c] for c in presentes]
    lodo[s] = f1_score(y, p, average='macro', labels=labs, zero_division=0)
    print(f'   F1-macro({s}) = {lodo[s]:.3f}')

## 7. O gap de generalização: *pooled* vs *LODO*

In [ ]:
f1_lodo = np.mean(list(lodo.values()))
print(f'F1 POOLED = {f1_pooled:.3f}  |  F1 LODO (media) = {f1_lodo:.3f}  |  GAP = {f1_pooled - f1_lodo:.3f}')
plt.bar(['pooled'] + [f'LODO\n{s}' for s in lodo], [f1_pooled] + list(lodo.values()))
plt.ylabel('F1-macro'); plt.title('Otimista (pooled) vs realista (LODO)'); plt.show()

## 8. Grad-CAM — o modelo olhou a lesão ou o fundo?
Usa o modelo *pooled*. Se o mapa de calor cair no **fundo/borda** em vez da **lesão**, é a
evidência visual da hipótese (o modelo se apoia na fonte).

In [ ]:
!pip install -q grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
amostra = va.groupby('label', group_keys=False).apply(lambda g: g.sample(1, random_state=0))
fig, ax = plt.subplots(1, len(amostra), figsize=(3 * len(amostra), 3))
for a, (_, r) in zip(np.atleast_1d(ax), amostra.iterrows()):
    img = Image.open(r['image_id']).convert('RGB').resize((224, 224))
    rgb = np.array(img) / 255.0
    g = cam(input_tensor=tf_eval(img).unsqueeze(0).to(DEV))[0]   # mesmo dispositivo do modelo
    a.imshow(show_cam_on_image(rgb, g, use_rgb=True)); a.set_title(r['label']); a.axis('off')
plt.show()

## 9. Salvar o modelo treinado no Drive

In [ ]:
torch.save({'state_dict': model.state_dict(), 'classes': CLASSES},
           '/content/drive/MyDrive/banana_pad/model_resnet18.pth')
print('modelo salvo no Drive: banana_pad/model_resnet18.pth')